# 로봇 제어 실습

쓰는 명령은 `Arm`과 `load_pick_targets`입니다.

프로젝트 폴더에서 엽니다. 커널은 `lerobot`입니다. 칸은 위에서 아래로 한 번씩 실행합니다. 팔 켜기 칸을 두 번 실행하면 팔이 겹칩니다.

## 시작 전에

- 펜던트는 끄고 진행합니다. 이 노트북이 3D 화면(Meshcat)을 띄웁니다. 브라우저가 안 열리면 출력된 주소를 엽니다.
- 좌표는 로봇 기준입니다. +X 앞, +Y 왼쪽, +Z 위. 위치는 mm, 자세는 도입니다.
- `grip(0)`은 닫힘, `grip(100)`은 완전히 열림입니다.
- 이동 명령은 도착할 때까지 그 칸에서 기다립니다.
- `"rel"`은 지금 위치에 더합니다. `"abs"`는 베이스 기준 목표입니다.
- `rpy`는 더하는 각도가 아니라 목표 자세입니다.

## 1. 팔 켜기

`where()`는 지금 관절, 그리퍼, 끝 위치(`xyz_mm`), 자세(`rpy_deg`)를 돌려줍니다.

In [1]:
from motion import Arm, load_pick_targets

arm = Arm()
print(arm.where())

Arm      mode=virtual
Meshcat  http://127.0.0.1:7000/static/
{'joints_deg': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 'gripper': 50.0, 'xyz_mm': [351.84, -0.5, 224.17], 'rpy_deg': [-180.0, -87.21, -90.0]}


## 2. 시작 자세와 그리퍼

`initial()`은 작업을 시작하는 자세입니다. 그리퍼는 닫혀 있습니다.

In [ ]:
arm.initial()
arm.grip(100)
print(arm.where())

## 3. 상대 이동

앞 30mm, 그다음 왼쪽 20mm와 위 15mm입니다. `speed`를 생략하면 25 mm/s입니다.

In [ ]:
arm.moveL([30, 0, 0], "rel", speed=10)
arm.moveL([0, 20, 15], "rel")
print(arm.where()["xyz_mm"])

## 4. 절대 위치와 자세

두 번째 이동은 위치와 손목 자세를 함께 지정합니다.

In [ ]:
arm.moveL([200, 0, 120], "abs")
arm.moveL([180, 40, 90], "abs", rpy=[0, 80, 0])
print(arm.where())

## 5. 지금 높이를 읽어서 올리기

In [ ]:
here = arm.where()
x, y, z = here["xyz_mm"]
arm.moveL([x, y, z + 40], "abs")
print(arm.where()["xyz_mm"])

## 6. 관절

`moveJ`는 관절을 돌립니다. 그리퍼 끝은 직선으로 가지 않습니다. `gripper=20`은 그리퍼를 지금보다 20만큼 더 엽니다.

In [ ]:
arm.moveJ([0, -40, 40, 0, -90, 0], "abs")
arm.moveJ([0, 10, 0, 0, 0, 0], "rel", gripper=20)
print(arm.where())

## 7. 홈

`home()`은 관절이 0인 L자입니다. 그리퍼는 반쯤 열립니다.

In [ ]:
arm.home()
print(arm.where())

## 8. 좌표를 적어서 집기

카메라는 쓰지 않습니다. 순서는 열기, 내리기, 닫기, 들기, 옆으로 옮기기, 내리기, 열기, 빠지기입니다.

In [ ]:
arm.initial()
arm.grip(100)
arm.moveL([0, 0, -40], "rel")
arm.grip(0)
arm.moveL([0, 0, 50], "rel")
arm.moveL([0, 80, 0], "rel")
arm.moveL([0, 0, -40], "rel")
arm.grip(100)
arm.moveL([0, 0, 40], "rel")
arm.home()

## 9. 카메라로 집기

이 칸을 실행하기 전에 다른 터미널에서 `python yolo/pose/roi_cloud.py`를 켜 둡니다. 브라켓이 화면에 보여야 합니다.

`load_pick_targets(arm)`이 대기 위치와 집기 위치를 채웁니다. 내려놓는 X, Y는 `drop_x`, `drop_y`입니다.

In [ ]:
drop_x, drop_y = 150.0, -100.0
pick = load_pick_targets(arm)
z_g = pick.g_xyz[2]

arm.initial()
arm.moveL(pick.p_xyz, "abs", rpy=pick.p_rpy)
arm.grip(30)
arm.moveL(pick.g_xyz, "abs", rpy=pick.g_rpy)
arm.grip(0)
arm.moveL([0, 0, 30], "rel")
arm.moveL([drop_x, drop_y, z_g + 50], "abs", rpy=pick.place_rpy, path="joint")
arm.moveL([drop_x, drop_y, z_g], "abs", rpy=pick.place_rpy)
arm.grip(30)
arm.moveL([0, 0, 50], "rel")
arm.initial()

## 10. 끄기

다시 움직이려면 커널을 재시작하고 1번부터 실행합니다.

In [ ]:
arm.close()

## 11. 실기

강사가 실기라고 한 뒤에만 합니다. 위의 가상 팔을 끈 다음, `REAL = True`로 바꾸고 이 칸만 실행합니다.

In [ ]:
REAL = False

if REAL:
    arm = Arm(mode="real")
    print(arm.where())
    arm.initial()
    arm.close()
else:
    print("실기 시간이 아니면 이 칸은 넘어갑니다.")